# Vertex AI MLOps End-to-End Demo — v1 → v2 Model Versioning

This notebook demonstrates an end-to-end MLOps workflow on **Vertex AI** (Gemini Enterprise
Agent Platform), using a **Custom Training Job** to train, register, deploy, and retire two
versions of the same model — the exact pattern a real team uses when they retrain a model on
more data / better hyperparameters and need to roll the new version out safely.

> **Goal:** Predict whether a customer will buy a product using a tiny synthetic dataset, while
> walking through every MLOps step a production model goes through — including canary traffic
> splitting between **v1** and **v2**, and retiring v1 once v2 is trusted.

## What this notebook does

1. Authenticate to Google Cloud
2. Ask for Project ID, Region, Bucket, and Model display name
3. Enable required APIs
4. Create a Cloud Storage bucket
5. Create two synthetic datasets — a small "launch" dataset and a larger "retrain" dataset
6. Build a parameterized trainer package and upload it as a source distribution to Cloud Storage
7. Submit a **Custom Training Job** that trains and registers **Model v1**
8. Submit a second **Custom Training Job** that trains and registers **Model v2** as a **new
   version of the same model** (not a separate model)
9. Create an **Endpoint** and deploy v1 at 100% traffic
10. **Canary-deploy** v2 alongside v1 with a traffic split, and observe the split empirically via
    online predictions
11. **Promote** v2 to 100% traffic by retiring v1
12. Reproduce the entire train/register/deploy flow as one repeatable **Vertex AI Pipeline**
13. **Clean up** every billable resource created (endpoint, model, bucket)

Every major milestone has a **🖥️ Console Demo** note telling you exactly where to look in the
Google Cloud Console to show the class what just happened.

**Note:** The authenticated user needs Vertex AI Admin, Storage Admin, and Service Usage Admin
(or equivalent) IAM roles for this notebook to run end-to-end.


In [ ]:
# ===========================
# USER INPUTS
# ===========================
PROJECT_ID = input("Enter GCP Project ID: ").strip()
REGION = input("Enter Region (default us-central1): ").strip() or "us-central1"
BUCKET_NAME = input("Bucket name (leave blank to auto-create): ").strip()
MODEL_DISPLAY_NAME = input("Model display name (default customer-buy-predictor): ").strip() or "customer-buy-predictor"

if not BUCKET_NAME:
    BUCKET_NAME = f"{PROJECT_ID}-vertex-mlops-demo"

PIPELINE_ROOT = f"gs://{BUCKET_NAME}/pipeline_root"

print("Project:      ", PROJECT_ID)
print("Region:       ", REGION)
print("Bucket:       ", BUCKET_NAME)
print("Model name:   ", MODEL_DISPLAY_NAME)
print("Pipeline root:", PIPELINE_ROOT)


## Authenticate

In [ ]:
IN_COLAB = "google.colab" in str(get_ipython())

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated in Colab.")
else:
    print("Not running in Colab — assuming 'gcloud auth application-default login' has already been run.")


In [ ]:
!pip -q install --upgrade google-cloud-aiplatform "google-cloud-pipeline-components>=2.0" \
    kfp==2.8.0 google-cloud-storage scikit-learn pandas joblib setuptools wheel


## Configure gcloud

In [ ]:
!gcloud config set project $PROJECT_ID
!gcloud config set ai/region $REGION


## Enable required APIs

In [ ]:
!gcloud services enable \
  aiplatform.googleapis.com \
  artifactregistry.googleapis.com \
  storage.googleapis.com \
  cloudbuild.googleapis.com \
  compute.googleapis.com


## Create Bucket If Needed

**Why the `google-cloud-storage` client instead of `!gsutil`:** in some Colab / Python 3.12
runtimes, `gsutil`'s launcher script conflicts with an old system `pkg_resources` and throws an
`AttributeError: module 'pkgutil' has no attribute 'ImpImporter'` traceback — a known compatibility
issue, not something wrong with your project or account. The command sometimes still completes
despite the noisy traceback, but it's not reliable across environments. Using the
`google-cloud-storage` Python client sidesteps `gsutil`'s launcher entirely, so every bucket/object
operation in this notebook uses it instead of shelling out.


In [ ]:
from google.cloud import storage

storage_client = storage.Client(project=PROJECT_ID)

existing_bucket = storage_client.lookup_bucket(BUCKET_NAME)
if existing_bucket is None:
    bucket = storage_client.create_bucket(BUCKET_NAME, location=REGION)
    print(f"Created bucket: {bucket.name}")
else:
    bucket = existing_bucket
    print("Bucket already exists")


## Initialize Vertex AI SDK

In [ ]:
import vertexai
from google.cloud import aiplatform

vertexai.init(project=PROJECT_ID, location=REGION, staging_bucket=f"gs://{BUCKET_NAME}")
aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=f"gs://{BUCKET_NAME}")
print("Vertex AI SDK initialized.")


## 1. Two Datasets — a "Launch" Set and a "Retrain" Set

**Why two datasets:** the whole point of this notebook is to demonstrate a realistic reason a
**v2** model gets built — not just a code change, but a retrain on more data collected after
launch. Both datasets share the same schema (`age`, `income` → `buy`) so the two model versions
stay drop-in compatible behind the same online-prediction endpoint.

- `customer_v1.csv` — 50 rows, standing in for the small dataset available at initial launch.
- `customer_v2.csv` — 200 rows, standing in for the larger dataset available a few weeks later.


In [ ]:
import pandas as pd
from random import randint, seed

seed(42)
rows = []
for i in range(50):
    age = randint(18, 60)
    income = randint(20000, 100000)
    buy = 1 if (income > 50000 and age > 25) else 0
    rows.append([age, income, buy])

df_v1 = pd.DataFrame(rows, columns=["age", "income", "buy"])
df_v1.to_csv("customer_v1.csv", index=False)

GCS_DATA_V1 = f"gs://{BUCKET_NAME}/data/customer_v1.csv"
bucket.blob("data/customer_v1.csv").upload_from_filename("customer_v1.csv")

print(f"{len(df_v1)} rows uploaded to {GCS_DATA_V1}")
df_v1.head(3)


In [ ]:
seed(7)
rows = []
for i in range(200):
    age = randint(18, 65)
    income = randint(18000, 120000)
    buy = 1 if (income > 50000 and age > 25) else 0
    rows.append([age, income, buy])

df_v2 = pd.DataFrame(rows, columns=["age", "income", "buy"])
df_v2.to_csv("customer_v2.csv", index=False)

GCS_DATA_V2 = f"gs://{BUCKET_NAME}/data/customer_v2.csv"
bucket.blob("data/customer_v2.csv").upload_from_filename("customer_v2.csv")

print(f"{len(df_v2)} rows uploaded to {GCS_DATA_V2}")
df_v2.head(3)


## 2. A Parameterized Trainer Package

The trainer script now accepts hyperparameters (`--n-estimators`, `--max-depth`) as command-line
arguments instead of hardcoding them. This is what lets the **same package** train both v1
(smaller, shallower forest on the launch data) and v2 (larger, deeper forest on the retrain data)
— exactly how a real team would iterate without maintaining two copies of the training code.

It also reads `AIP_MODEL_DIR` — the environment variable Vertex AI automatically injects into the
training container when you register a model straight from a training job — falling back to a
`--model-dir` argument for local testing.

**Important:** `AIP_MODEL_DIR` is a `gs://` URI, not a local filesystem path. `joblib.dump()` and
`os.makedirs()` only understand local paths, so the script always writes its output locally first,
then explicitly uploads to GCS with the `google-cloud-storage` client (already included in the
prebuilt sklearn training image) whenever `--model-dir` starts with `gs://`. Skipping this step is
a common silent failure: training appears to succeed, but Vertex AI's model-registration step then
fails with `There are no files under "gs://.../model" to copy` because nothing was ever actually
written to Cloud Storage.


In [ ]:
import os, textwrap

os.makedirs("trainer", exist_ok=True)
open("trainer/__init__.py", "w").close()

task_code = textwrap.dedent('''
    import argparse
    import json
    import os

    import joblib
    import pandas as pd
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score
    from sklearn.model_selection import train_test_split


    def upload_file_to_gcs(local_path, gcs_uri):
        # Upload a local file to a gs://bucket/path URI using the storage client.
        from google.cloud import storage

        assert gcs_uri.startswith("gs://"), f"Not a GCS URI: {gcs_uri}"
        bucket_name, blob_path = gcs_uri[len("gs://"):].split("/", 1)
        client = storage.Client()
        bucket = client.bucket(bucket_name)
        bucket.blob(blob_path).upload_from_filename(local_path)


    parser = argparse.ArgumentParser()
    parser.add_argument("--data", required=True, help="GCS or local path to training CSV")
    parser.add_argument("--model-dir", default=os.environ.get("AIP_MODEL_DIR", "model_output"))
    parser.add_argument("--n-estimators", type=int, default=100)
    parser.add_argument("--max-depth", type=int, default=None)
    args = parser.parse_args()

    print(f"Loading data from {args.data}")
    df = pd.read_csv(args.data)
    X = df[["age", "income"]]
    y = df["buy"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = RandomForestClassifier(
        n_estimators=args.n_estimators,
        max_depth=args.max_depth,
        random_state=42,
    )
    model.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    print(f"n_estimators={args.n_estimators} max_depth={args.max_depth}")
    print(f"train_accuracy={train_acc:.4f} test_accuracy={test_acc:.4f}")

    # Always write locally first -- AIP_MODEL_DIR / --model-dir may be a gs:// URI,
    # which joblib.dump() and os.makedirs() cannot write to directly.
    local_dir = "local_model_output"
    os.makedirs(local_dir, exist_ok=True)
    local_model_path = os.path.join(local_dir, "model.joblib")
    local_metrics_path = os.path.join(local_dir, "metrics.json")

    joblib.dump(model, local_model_path)
    with open(local_metrics_path, "w") as f:
        json.dump(
            {
                "n_estimators": args.n_estimators,
                "max_depth": args.max_depth,
                "train_accuracy": train_acc,
                "test_accuracy": test_acc,
                "rows_trained_on": len(df),
            },
            f,
        )

    if args.model_dir.startswith("gs://"):
        model_dir = args.model_dir.rstrip("/")
        upload_file_to_gcs(local_model_path, f"{model_dir}/model.joblib")
        upload_file_to_gcs(local_metrics_path, f"{model_dir}/metrics.json")
        print(f"Uploaded model and metrics to {model_dir}")
    else:
        os.makedirs(args.model_dir, exist_ok=True)
        joblib.dump(model, os.path.join(args.model_dir, "model.joblib"))
        with open(os.path.join(args.model_dir, "metrics.json"), "w") as f:
            json.dump(
                {
                    "n_estimators": args.n_estimators,
                    "max_depth": args.max_depth,
                    "train_accuracy": train_acc,
                    "test_accuracy": test_acc,
                    "rows_trained_on": len(df),
                },
                f,
            )
        print(f"Model and metrics written to {args.model_dir}")

    print("Training complete.")
''')

with open("trainer/task.py", "w") as f:
    f.write(task_code)

print("trainer/task.py written.")


## 3. Package the Trainer and Upload It to Cloud Storage

`CustomPythonPackageTrainingJob` needs the trainer code as a **Python source distribution**
(a `.tar.gz`) sitting in Cloud Storage — Vertex AI installs it inside the training container at
job start. This is the piece the original notebook flagged as a "next step"; it's built here with
plain `setuptools`.


In [ ]:
setup_py = textwrap.dedent('''
    from setuptools import setup, find_packages

    setup(
        name="trainer",
        version="0.1",
        packages=find_packages(),
        include_package_data=True,
        install_requires=["scikit-learn", "pandas", "joblib"],
    )
''')

with open("setup.py", "w") as f:
    f.write(setup_py)

!python setup.py sdist --formats=gztar

TRAINER_PACKAGE_URI = f"gs://{BUCKET_NAME}/trainer/trainer-0.1.tar.gz"
bucket.blob("trainer/trainer-0.1.tar.gz").upload_from_filename("dist/trainer-0.1.tar.gz")

print("Trainer package uploaded to:", TRAINER_PACKAGE_URI)


## 4. Train & Register Model v1

This single call does two jobs at once: it runs a **Custom Training Job** on the launch dataset,
*and* — because `model_display_name` and `model_serving_container_image_uri` are supplied —
registers the resulting model artifact directly into the **Vertex AI Model Registry** as version
1, with no separate upload step needed.

**A note on the container image URIs below**, since it's an easy trap: the **training** repo
(`.../training/...`) and the **prediction** repo (`.../prediction/...`) don't share a version
lineup or even a naming prefix — e.g. training only has `sklearn-cpu.1-0` and `sklearn-cpu.1-6`
(no `1-5`), while prediction has `1-4`, `1-5`, and `1-6`. Always check Google's
[supported frameworks list](https://docs.cloud.google.com/vertex-ai/docs/supported-frameworks-list)
for the current, real image names rather than assuming the two repos mirror each other — and keep
the scikit-learn version number **identical** on both sides, since a training/serving version
mismatch causes its own failure at prediction time (`Trying to unpickle estimator ... from version
X when using version Y`). Only the `latest` tag is supported on any of these images.


In [ ]:
job_v1 = aiplatform.CustomPythonPackageTrainingJob(
    display_name="customer-buy-training-v1",
    python_package_gcs_uri=TRAINER_PACKAGE_URI,
    python_module_name="trainer.task",
    container_uri="us-docker.pkg.dev/vertex-ai/training/sklearn-cpu.1-6:latest",
    model_serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-6:latest",
)

model_v1 = job_v1.run(
    args=[
        "--data", GCS_DATA_V1,
        "--n-estimators", "50",
        "--max-depth", "3",
    ],
    base_output_dir=f"gs://{BUCKET_NAME}/models/v1",
    model_display_name=MODEL_DISPLAY_NAME,
    model_version_aliases=["v1", "baseline"],
    model_version_description="Baseline model trained on the 50-row launch dataset.",
    replica_count=1,
    machine_type="n1-standard-4",
    sync=True,
)

print("Model v1 registered:")
print("  resource_name:", model_v1.resource_name)
print("  version_id:   ", model_v1.version_id)


> 🖥️ **Check it in the console:** **Vertex AI (Agent Platform) → Training → Custom jobs** —
> `customer-buy-training-v1` shows as Succeeded; open it and click **View logs** to show the
> class the printed `train_accuracy` / `test_accuracy` line straight from the trainer script.
> Then go to **Vertex AI → Model Registry** — the model `customer-buy-predictor` now appears
> with a single version, **Version 1**, tagged with the `v1` / `baseline` aliases you just set.


## 5. Train & Register Model v2 — as a New Version of the *Same* Model

Same trainer package, different hyperparameters (a bigger, deeper forest), trained on the larger
retrain dataset. The key difference from Section 4: **`parent_model`** is set to `model_v1`'s
resource name, and **`is_default_version=False`**. This tells Vertex AI to register the result as
**version 2 of the existing model**, not a brand-new model resource — exactly the Model Registry
versioning feature you'd use for any real retrain.


In [ ]:
job_v2 = aiplatform.CustomPythonPackageTrainingJob(
    display_name="customer-buy-training-v2",
    python_package_gcs_uri=TRAINER_PACKAGE_URI,
    python_module_name="trainer.task",
    container_uri="us-docker.pkg.dev/vertex-ai/training/sklearn-cpu.1-6:latest",
    model_serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-6:latest",
)

model_v2 = job_v2.run(
    args=[
        "--data", GCS_DATA_V2,
        "--n-estimators", "200",
        "--max-depth", "6",
    ],
    base_output_dir=f"gs://{BUCKET_NAME}/models/v2",
    model_display_name=MODEL_DISPLAY_NAME,
    parent_model=model_v1.resource_name,
    is_default_version=False,
    model_version_aliases=["v2", "challenger"],
    model_version_description="Retrained on the 200-row dataset with a larger, deeper forest.",
    replica_count=1,
    machine_type="n1-standard-4",
    sync=True,
)

print("Model v2 registered as a new version of the same model:")
print("  resource_name:", model_v2.resource_name)
print("  version_id:   ", model_v2.version_id)


In [ ]:
# List every version registered under this model resource
print(f"All versions of '{MODEL_DISPLAY_NAME}':\n")
for v in aiplatform.Model(model_v1.resource_name).versioning_registry.list_versions():
    print(f"  version_id={v.version_id}  aliases={v.version_aliases}  created={v.version_create_time}")


> 🖥️ **Check it in the console:** **Vertex AI → Model Registry → `customer-buy-predictor`** —
> click into the model and open the **Versions** tab. You'll now see **Version 1** (`v1` /
> `baseline`) and **Version 2** (`v2` / `challenger`) both listed under the same model, with their
> individual descriptions, creation times, and training job links. This is the single screen that
> best tells the "same model, evolving over time" story — better than two separate model cards
> would.


## 6. Create an Endpoint and Deploy v1 at 100% Traffic

Before any retraining story matters, v1 needs to actually be serving traffic. This mirrors a
normal initial production rollout.


In [ ]:
endpoint = aiplatform.Endpoint.create(display_name=f"{MODEL_DISPLAY_NAME}-endpoint")

endpoint.deploy(
    model=model_v1,
    deployed_model_display_name="customer-buy-v1",
    machine_type="n1-standard-2",
    traffic_percentage=100,
    sync=True,
)

print("Endpoint created and v1 deployed at 100% traffic.")
print("Endpoint resource name:", endpoint.resource_name)


> 🖥️ **Check it in the console:** **Vertex AI → Online prediction → Endpoints** —
> `customer-buy-predictor-endpoint` now shows one deployed model, `customer-buy-v1`, at **100%**
> traffic.


## 7. Canary-Deploy v2 Alongside v1

`traffic_percentage=20` deploys v2 onto the *same* endpoint and automatically rebalances the
existing deployment down to 80% — a live canary rollout, with both versions serving real traffic
side by side.


In [ ]:
endpoint.deploy(
    model=model_v2,
    deployed_model_display_name="customer-buy-v2",
    machine_type="n1-standard-2",
    traffic_percentage=20,
    sync=True,
)

print("v2 canary-deployed. Current traffic split:")
for dm in endpoint.list_models():
    pct = endpoint.traffic_split.get(dm.id, 0)
    print(f"  {dm.display_name}  (id={dm.id})  ->  {pct}%")


> 🖥️ **Check it in the console:** **Vertex AI → Online prediction → Endpoints →
> `customer-buy-predictor-endpoint`** — the **Deployed models** panel now shows two rows,
> `customer-buy-v1` and `customer-buy-v2`, with a visible **80 / 20** traffic split. This is the
> best live moment in the whole demo to pause and let the class absorb what canary deployment
> actually looks like in the console, rather than just as a concept on a slide.


TEST the prediction

In [ ]:
from typing import Dict, List, Union
from google.cloud import aiplatform
from google.protobuf import json_format
from google.protobuf.struct_pb2 import Value


def predict_custom_trained_model_sample(
    project: str,
    endpoint_id: str,
    instances: Union[Dict, List],
    location: str = "us-central1",
    api_endpoint: str = "us-central1-aiplatform.googleapis.com",
):
    client_options = {"api_endpoint": api_endpoint}
    client = aiplatform.gapic.PredictionServiceClient(client_options=client_options)
    instances = instances if isinstance(instances, list) else [instances]
    instances = [json_format.ParseDict(i, Value()) for i in instances]
    parameters = json_format.ParseDict({}, Value())
    endpoint = client.endpoint_path(project=project, location=location, endpoint=endpoint_id)
    response = client.predict(endpoint=endpoint, instances=instances, parameters=parameters)
    print("deployed_model_id:", response.deployed_model_id)
    for prediction in response.predictions:
        print("prediction:", prediction)
    return response


predict_custom_trained_model_sample(
    project="1002412473126",
    endpoint_id="6018911918155104256",
    location="us-central1",
    instances=[[35, 62000], [22, 25000], [45, 80000]],
)

## 8. Online Prediction — Watch the Traffic Split in Action

Every prediction response includes `model_version_id`, so calling the endpoint repeatedly and
tallying which version answered each time gives an empirical view of the traffic split — not just
a console screenshot claim.


In [ ]:
from collections import Counter

sample_customers = pd.DataFrame({
    "age":    [22, 30, 45, 29, 52, 19, 38, 41, 26, 60],
    "income": [25000, 60000, 80000, 45000, 95000, 21000, 52000, 30000, 71000, 40000],
})

version_tally = Counter()
for _, row in sample_customers.iterrows():
    resp = endpoint.predict(instances=[[int(row["age"]), int(row["income"])]])
    version_tally[resp.model_version_id] += 1
    print(f"age={row['age']:>3}  income={row['income']:>6}  "
          f"prediction={resp.predictions[0]}  served_by_version={resp.model_version_id}")

print("\nObserved traffic split across", len(sample_customers), "calls:", dict(version_tally))
print("(With only 10 calls this will be noisy around 80/20 — run the cell again, or with more")
print("rows in sample_customers, for a split that converges closer to the configured percentage.)")


## 9. Promote v2 — Retire v1, Move to 100% Traffic

Once the canary looks healthy, the standard next move is to retire the old version rather than
juggle a manual traffic-split update: **undeploying v1 automatically routes 100% of traffic to
whatever's left on the endpoint** — v2.


In [ ]:
for dm in endpoint.list_models():
    if dm.display_name == "customer-buy-v1":
        print(f"Undeploying v1 (deployed_model_id={dm.id}) — promoting v2 to 100% traffic")
        endpoint.undeploy(deployed_model_id=dm.id, sync=True)

print("\nCurrent traffic split:")
for dm in endpoint.list_models():
    pct = endpoint.traffic_split.get(dm.id, 0)
    print(f"  {dm.display_name}  (id={dm.id})  ->  {pct}%")


In [ ]:
version_tally = Counter()
for _, row in sample_customers.iterrows():
    resp = endpoint.predict(instances=[[int(row["age"]), int(row["income"])]])
    version_tally[resp.model_version_id] += 1

print("Traffic split after promotion:", dict(version_tally_after))




> 🖥️ **Check it in the console:** back on the endpoint's **Deployed models** panel,
> `customer-buy-v1` is gone and `customer-buy-v2` now shows **100%**. On **Model Registry →
> `customer-buy-predictor` → Versions**, both versions still exist — undeploying only removes a
> version from serving traffic, it doesn't delete the version record, which is exactly the
> rollback safety net you'd want in production (you could redeploy v1 in seconds if v2 turned out
> to have a problem).


## 10. Vertex AI Pipelines — Reproducing the v1/v2 Workflow as One Repeatable Job

Everything in Sections 4–7 was run as individual notebook cells so each step could be inspected
along the way. In production, you'd wrap that same train → register → deploy sequence into a
**Vertex AI Pipeline** so it runs as one repeatable, scheduled, auditable job instead of manual
cells — this section builds that pipeline for real, using prebuilt
`google_cloud_pipeline_components` (GCPC) ops, not a placeholder.

**What this pipeline actually does, end to end:**
1. `CustomTrainingJobOp` trains v1 on the launch dataset (same trainer package, same
   hyperparameters as Section 4).
2. A `dsl.importer` node picks up the resulting model artifact from GCS and wraps it as an
   `UnmanagedContainerModel`, then `ModelUploadOp` registers it as version 1 of a model.
3. `CustomTrainingJobOp` trains v2 on the retrain dataset (Section 5's hyperparameters).
4. `ModelUploadOp` registers v2 as a **new version of the same model**, via `parent_model` —
   the pipeline-native equivalent of the SDK's `parent_model=` argument used earlier.
5. `EndpointCreateOp` creates a fresh endpoint.
6. `ModelDeployOp` deploys v1 live at **100% traffic**.
7. `ModelDeployOp` deploys v2 to the same endpoint as a **shadow deployment at 0% traffic** —
   it's live and servable, but doesn't receive real traffic yet.

**Why v2 is shadow-deployed rather than canaried inside the pipeline:** setting an exact traffic
split across two *separately* deployed models needs the first model's live `deployed_model_id`,
which isn't cleanly available as a typed value inside the pipeline graph without extra plumbing.
More importantly, promoting a new version to receive real traffic is a **deliberate, gated
decision** — not something you want an unattended pipeline run to do by itself. So this pipeline
automates the part that's safe to fully automate (train, register, get v2 warm and ready to
serve), and leaves the actual cutover as the same manual action already demonstrated in Section 9
(`endpoint.undeploy()` on the old version, which auto-promotes whatever's left to 100%) — just
pointed at this pipeline's endpoint instead.

This uses a **separate model and endpoint** from the ones created manually in Sections 4–9
(`-pipeline` suffixed), so the two approaches can be compared side by side in the console without
colliding.


In [ ]:
!gcloud services enable compute.googleapis.com --project=himanshugcpproject1

!gcloud projects add-iam-policy-binding himanshugcpproject1 --member="serviceAccount:1002412473126-compute@developer.gserviceaccount.com" --role="roles/owner"

In [ ]:
from google_cloud_pipeline_components.v1.custom_job import CustomTrainingJobOp
from google_cloud_pipeline_components.v1.model import ModelUploadOp
from google_cloud_pipeline_components.v1.endpoint import EndpointCreateOp, ModelDeployOp
from google_cloud_pipeline_components.types import artifact_types
from kfp import dsl, compiler
from kfp.dsl import importer

PIPELINE_MODEL_DISPLAY_NAME = f"{MODEL_DISPLAY_NAME}-pipeline"
PIPELINE_ENDPOINT_DISPLAY_NAME = f"{MODEL_DISPLAY_NAME}-pipeline-endpoint"


@dsl.pipeline(name="customer-buy-v1-v2-pipeline")
def v1_v2_pipeline(
    project: str,
    location: str,
    worker_pool_specs_v1: list,
    worker_pool_specs_v2: list,
    base_output_dir_v1: str,
    base_output_dir_v2: str,
    model_dir_v1: str,
    model_dir_v2: str,
    serving_container_uri: str,
    model_display_name: str,
    endpoint_display_name: str,
):
    # --- Train + register v1 ---
    train_v1 = CustomTrainingJobOp(
        project=project,
        location=location,
        display_name="customer-buy-training-v1-pipeline",
        worker_pool_specs=worker_pool_specs_v1,
        base_output_directory=base_output_dir_v1,
    )

    import_v1 = importer(
        artifact_uri=model_dir_v1,
        artifact_class=artifact_types.UnmanagedContainerModel,
        metadata={"containerSpec": {"imageUri": serving_container_uri}},
    ).after(train_v1)

    upload_v1 = ModelUploadOp(
        project=project,
        location=location,
        display_name=model_display_name,
        unmanaged_container_model=import_v1.outputs["artifact"],
        version_aliases=["v1", "baseline"],
        description="Baseline model trained on the 50-row launch dataset (via pipeline).",
    )

    # --- Train + register v2 as a new version of the same model ---
    train_v2 = CustomTrainingJobOp(
        project=project,
        location=location,
        display_name="customer-buy-training-v2-pipeline",
        worker_pool_specs=worker_pool_specs_v2,
        base_output_directory=base_output_dir_v2,
    )

    import_v2 = importer(
        artifact_uri=model_dir_v2,
        artifact_class=artifact_types.UnmanagedContainerModel,
        metadata={"containerSpec": {"imageUri": serving_container_uri}},
    ).after(train_v2)

    upload_v2 = ModelUploadOp(
        project=project,
        location=location,
        display_name=model_display_name,
        parent_model=upload_v1.outputs["model"],
        unmanaged_container_model=import_v2.outputs["artifact"],
        version_aliases=["v2", "challenger"],
        description="Retrained on the 200-row dataset with a larger, deeper forest (via pipeline).",
    )

    # --- Endpoint: v1 live at 100%, v2 shadow-deployed at 0% traffic ---
    endpoint_create = EndpointCreateOp(
        project=project,
        location=location,
        display_name=endpoint_display_name,
    )

    deploy_v1 = ModelDeployOp(
        model=upload_v1.outputs["model"],
        endpoint=endpoint_create.outputs["endpoint"],
        deployed_model_display_name="customer-buy-v1-pipeline",
        dedicated_resources_machine_type="n1-standard-2",
        dedicated_resources_min_replica_count=1,
        dedicated_resources_max_replica_count=1,
        traffic_split={"0": 100},
    )

    # No traffic_split specified here -> v2 is deployed but the existing split (v1 @ 100%) is
    # left untouched, so v2 receives 0% traffic: a shadow deployment, not a canary.
    deploy_v2 = ModelDeployOp(
        model=upload_v2.outputs["model"],
        endpoint=endpoint_create.outputs["endpoint"],
        deployed_model_display_name="customer-buy-v2-pipeline",
        dedicated_resources_machine_type="n1-standard-2",
        dedicated_resources_min_replica_count=1,
        dedicated_resources_max_replica_count=1,
    ).after(deploy_v1)


# Build the worker pool specs as plain Python data *before* compiling the pipeline -- this keeps
# the pipeline definition itself free of any string-concatenation of pipeline parameters, which
# KFP's DSL does not reliably support inside a @dsl.pipeline-decorated function body.
TRAIN_CONTAINER_URI = "us-docker.pkg.dev/vertex-ai/training/sklearn-cpu.1-6:latest"
SERVING_CONTAINER_URI = "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-6:latest"

worker_pool_specs_v1 = [{
    "machine_spec": {"machine_type": "n1-standard-4"},
    "replica_count": 1,
    "python_package_spec": {
        "executor_image_uri": TRAIN_CONTAINER_URI,
        "package_uris": [TRAINER_PACKAGE_URI],
        "python_module": "trainer.task",
        "args": [
            "--data", GCS_DATA_V1,
            "--n-estimators", "50",
            "--max-depth", "3",
            "--model-dir", f"gs://{BUCKET_NAME}/pipeline_models/v1/model",
        ],
    },
}]

worker_pool_specs_v2 = [{
    "machine_spec": {"machine_type": "n1-standard-4"},
    "replica_count": 1,
    "python_package_spec": {
        "executor_image_uri": TRAIN_CONTAINER_URI,
        "package_uris": [TRAINER_PACKAGE_URI],
        "python_module": "trainer.task",
        "args": [
            "--data", GCS_DATA_V2,
            "--n-estimators", "200",
            "--max-depth", "6",
            "--model-dir", f"gs://{BUCKET_NAME}/pipeline_models/v2/model",
        ],
    },
}]

compiler.Compiler().compile(pipeline_func=v1_v2_pipeline, package_path="pipeline_v1_v2.json")

pipeline_job = aiplatform.PipelineJob(
    display_name="customer-buy-v1-v2-pipeline-run",
    template_path="pipeline_v1_v2.json",
    pipeline_root=PIPELINE_ROOT,
    parameter_values={
        "project": PROJECT_ID,
        "location": REGION,
        "worker_pool_specs_v1": worker_pool_specs_v1,
        "worker_pool_specs_v2": worker_pool_specs_v2,
        "base_output_dir_v1": f"gs://{BUCKET_NAME}/pipeline_models/v1",
        "base_output_dir_v2": f"gs://{BUCKET_NAME}/pipeline_models/v2",
        "model_dir_v1": f"gs://{BUCKET_NAME}/pipeline_models/v1/model",
        "model_dir_v2": f"gs://{BUCKET_NAME}/pipeline_models/v2/model",
        "serving_container_uri": SERVING_CONTAINER_URI,
        "model_display_name": PIPELINE_MODEL_DISPLAY_NAME,
        "endpoint_display_name": PIPELINE_ENDPOINT_DISPLAY_NAME,
    },
)
pipeline_job.submit(
    service_account="1002412473126-compute@developer.gserviceaccount.com"
)

print("Pipeline submitted asynchronously.")
print("This single run reproduces: train v1 -> register v1 -> train v2 -> register as version 2")
print("of the same model -> create endpoint -> deploy v1 live at 100% -> shadow-deploy v2 at 0%.")
print("Expect roughly as long as Sections 4-7 combined (~30-45 min), since it repeats that work.")


**Optional: wait for it to finish.** The pipeline runs in the background so the rest of the
notebook isn't blocked. If you want this cell to pause until the pipeline completes before you
move on to cleanup, set `WAIT_FOR_PIPELINE = True` below — otherwise just check progress in the
console at your own pace.


In [ ]:
WAIT_FOR_PIPELINE = False  # set True to block here until the pipeline finishes

if WAIT_FOR_PIPELINE:
    pipeline_job.wait()
    print("Pipeline finished with state:", pipeline_job.state)
else:
    print("Not waiting. Check the console for status, or call pipeline_job.wait() in a later cell")
    print("before running the cleanup section, so the cleanup can find and remove its resources.")


> 🖥️ **Check it in the console:** **Vertex AI → Pipelines** —
> `customer-buy-v1-v2-pipeline-run` shows a real multi-node DAG: two training jobs, two importer
> nodes, two model uploads (with the version-2 upload linked to version 1 as its parent), an
> endpoint creation, and two deployments. Click into `customer-buy-training-v1-pipeline` or
> `-v2-pipeline` to see the same accuracy logs as Sections 4–5. Once it finishes, **Model
> Registry → `customer-buy-predictor-pipeline`** shows the same v1/v2 versioning story as the
> manually-created model, and **Online prediction → Endpoints →
> `customer-buy-predictor-pipeline-endpoint`** shows v1 at 100% and v2 sitting at 0% — ready to
> promote via the same `endpoint.undeploy()` pattern from Section 9, just pointed at this
> endpoint's deployed model IDs instead.


## 11. Cleanup

Unlike the pay-per-call evaluation notebook, **this notebook creates continuously-billing
resources** — a deployed endpoint (billed while running, regardless of traffic) and a Cloud
Storage bucket. Run every cell in this section to avoid ongoing charges.


In [ ]:
# 1. Undeploy every remaining model from the endpoint, then delete the endpoint
for dm in endpoint.list_models():
    print("Undeploying:", dm.display_name)
    endpoint.undeploy(deployed_model_id=dm.id, sync=True)

endpoint.delete()
print("Endpoint deleted.")


In [ ]:
# 2. Delete the model resource — this removes ALL versions (v1 and v2) together,
#    since they share one underlying Model Registry resource.
aiplatform.Model(model_v1.resource_name).delete()
print("Model resource deleted (both v1 and v2 versions removed).")


In [ ]:
# 2b. Also clean up the SEPARATE endpoint and model created by the Section 10 pipeline run.
#     Looked up by display name (rather than by Python object reference) since the pipeline may
#     have finished well after it was submitted -- this is safe to re-run at any point.
pipeline_model_display_name = f"{MODEL_DISPLAY_NAME}-pipeline"
pipeline_endpoint_display_name = f"{MODEL_DISPLAY_NAME}-pipeline-endpoint"

pipeline_endpoints = aiplatform.Endpoint.list(filter=f'display_name="{pipeline_endpoint_display_name}"')
for ep in pipeline_endpoints:
    for dm in ep.list_models():
        ep.undeploy(deployed_model_id=dm.id, sync=True)
    ep.delete()
    print(f"Deleted pipeline endpoint: {pipeline_endpoint_display_name}")

pipeline_models = aiplatform.Model.list(filter=f'display_name="{pipeline_model_display_name}"')
for m in pipeline_models:
    aiplatform.Model(m.resource_name).delete()
    print(f"Deleted pipeline model: {pipeline_model_display_name}")

if not pipeline_endpoints and not pipeline_models:
    print("No pipeline-created resources found -- either the pipeline hasn't finished yet")
    print("(re-run this cell once it has, or call pipeline_job.wait() first), or they were")
    print("already cleaned up.")


In [ ]:
# 3. Delete the Cloud Storage bucket and everything in it
#    (training data, trainer package, model artifacts, pipeline root).
#    Deleting blobs individually first avoids the object-count limit on bucket.delete(force=True).
bucket_to_delete = storage_client.bucket(BUCKET_NAME)
blobs = list(bucket_to_delete.list_blobs())
for blob in blobs:
    blob.delete()
bucket_to_delete.delete()

print(f"Deleted {len(blobs)} objects and the bucket.")


In [ ]:
# 4. Delete local scratch files created during this notebook
import shutil

for path in ["trainer", "dist", "trainer.egg-info", "customer_v1.csv", "customer_v2.csv",
             "setup.py", "pipeline_v1_v2.json"]:
    if os.path.isdir(path):
        shutil.rmtree(path, ignore_errors=True)
    elif os.path.isfile(path):
        os.remove(path)

print("Local scratch files cleaned up.")


> 🖥️ **Final console check:** **Vertex AI → Online prediction → Endpoints** and **Vertex AI →
> Model Registry** should both now be empty of anything created in this session. **Cloud Storage →
> Buckets** should no longer list `{BUCKET_NAME}`. The one thing that *does* stay behind
> intentionally: **Vertex AI → Training → Custom jobs** and **Vertex AI → Pipelines** keep their
> run history even after the resources they created are deleted — same as the evaluation
> notebook's Experiments screen, this is your audit trail of what was trained and when, at no
> ongoing cost.


## Summary — What This Demonstrated

- A **Custom Training Job**, run twice with different data and hyperparameters, registering two
  **versions of a single Model Registry entry** rather than two unrelated models.
- A **canary deployment**: both versions serving live traffic on one endpoint at a controlled
  split, confirmed empirically via `model_version_id` on prediction responses — not just asserted.
- A **safe promotion**: retiring the old version rather than deleting it, keeping a one-command
  rollback path available for as long as the version record exists in the Model Registry.
- A full **Vertex AI Pipeline** reproducing that same train → register (versioned) → deploy flow
  as one repeatable, auditable job — with v2 landing as a live shadow deployment, ready for the
  same deliberate promotion step demonstrated manually above.
- A complete **teardown** of every billable resource, with an explicit note on what's deliberately
  left behind (job and pipeline history) versus what must be deleted (endpoint, model, bucket).

This is the same "v1 vs v2, evidence not vibes" spirit as the GenAI evaluation lab — here applied
to a classic tabular model instead of an LLM, and to *deployment* safety rather than prompt/model
quality scoring.
